In [1]:
import sys
!{sys.executable} -m pip install qiskit qiskit-aer qiskit-algorithms qiskit-optimization matplotlib


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import qiskit
print("Qiskit version :", qiskit.__version__)

from qiskit_aer import AerSimulator
print("Simulateur Aer prêt !")
print("Tout est bon.")

Qiskit version : 2.6.2
Simulateur Aer prêt !
Tout est bon.


In [3]:
from qiskit_aer.primitives import Sampler

In [4]:
# ============================================================
# PROJET 2 : Optimisation de routes de livraison au Bénin
# Qiskit Fall Fest 2026 - IndabaX Benin - Sèmè City
# Auteur : ADJOGAN Senou Gbenoukpo
# ============================================================

from qiskit_optimization import QuadraticProgram
from qiskit_optimization.algorithms import MinimumEigenOptimizer
from qiskit_algorithms import NumPyMinimumEigensolver
import numpy as np

print("Bibliothèques chargées avec succès !")

# --- 1. Données concrètes béninoises ---
# Points : 0 = Dépôt (Sèmè City), 1 = Dantokpa, 2 = Porto-Novo, 3 = Ouidah
noms = ["Depot_Seme", "Dantokpa", "Porto_Novo", "Ouidah"]

# Matrice de distances approximatives (en km)
distances = np.array([
    [0,  8,  30, 40],   # depuis Depot_Seme
    [8,  0,  35, 45],   # depuis Dantokpa
    [30, 35, 0,  55],   # depuis Porto_Novo
    [40, 45, 55, 0 ]    # depuis Ouidah
])

n = len(noms)
print("\nMatrice de distances (km) :")
print(distances)

# --- 2. Formulation du problème TSP simplifié ---
qp = QuadraticProgram(name="TSP_Livraison_Benin")

# Variables binaires x_i_j = 1 si on va de i vers j
for i in range(n):
    for j in range(n):
        if i != j:
            qp.binary_var(name=f"x_{i}_{j}")

# Objectif : minimiser la distance totale
linear_dict = {}
for i in range(n):
    for j in range(n):
        if i != j:
            linear_dict[f"x_{i}_{j}"] = float(distances[i][j])

qp.minimize(linear=linear_dict)

# Contraintes : chaque ville a exactement 1 départ et 1 arrivée
for i in range(n):
    # Un seul arc sortant de i
    out_dict = {f"x_{i}_{j}": 1 for j in range(n) if j != i}
    qp.linear_constraint(linear=out_dict, sense="==", rhs=1, name=f"out_{i}")
    
    # Un seul arc entrant vers i
    in_dict = {f"x_{j}_{i}": 1 for j in range(n) if j != i}
    qp.linear_constraint(linear=in_dict, sense="==", rhs=1, name=f"in_{i}")

print("\nProblème formulé avec succès.")
print(f"Nombre de variables : {qp.get_num_binary_vars()}")

# --- 3. Résolution ---
exact_solver = MinimumEigenOptimizer(NumPyMinimumEigensolver())
result = exact_solver.solve(qp)

print("\n=== SOLUTION TROUVÉE ===")
print(result.prettyprint())

# --- 4. Reconstruction du trajet ---
print("\n" + "="*55)
print("TRAJET RECOMMANDÉ POUR LE LIVREUR")
print("="*55)

solution = result.variables_dict
arcs = []
for var_name, val in solution.items():
    if val > 0.5 and var_name.startswith("x_"):
        parts = var_name.split("_")
        i, j = int(parts[1]), int(parts[2])
        arcs.append((i, j))
        print(f"  {noms[i]:15} --> {noms[j]:15}   ({distances[i][j]} km)")

total = sum(distances[i][j] for i, j in arcs)
print(f"\nDistance totale du circuit : {total} km")
print("="*55)
print("Projet 2 terminé avec succès !")

Bibliothèques chargées avec succès !

Matrice de distances (km) :
[[ 0  8 30 40]
 [ 8  0 35 45]
 [30 35  0 55]
 [40 45 55  0]]

Problème formulé avec succès.
Nombre de variables : 12

=== SOLUTION TROUVÉE ===
objective function value: 126.0
variable values: x_0_1=1.0, x_0_2=0.0, x_0_3=0.0, x_1_0=1.0, x_1_2=0.0, x_1_3=0.0, x_2_0=0.0, x_2_1=0.0, x_2_3=1.0, x_3_0=0.0, x_3_1=0.0, x_3_2=1.0
status: SUCCESS

TRAJET RECOMMANDÉ POUR LE LIVREUR
  Depot_Seme      --> Dantokpa          (8 km)
  Dantokpa        --> Depot_Seme        (8 km)
  Porto_Novo      --> Ouidah            (55 km)
  Ouidah          --> Porto_Novo        (55 km)

Distance totale du circuit : 126 km
Projet 2 terminé avec succès !
